# TRM Latent-History Sudoku Study

Reproducible Colab/T4 workflow for B0/B1/B2/B3/P1. The 55-minute setting is a runtime cap, not a guarantee of convergence or completion. Scaled-model inference is useful for iteration but is not publication-scale evidence.

In [ ]:
# Runtime → T4 GPU. Always clone the study branch (default main does not have it).
import os, pathlib, sys

REPO_URL = "https://github.com/Iliyabr/trm-latent-history-attention.git"
BRANCH = "feature/latent-history-attention"
ROOT = pathlib.Path("/content/trm-latent-history-attention")

if pathlib.Path("/content").exists():
    %cd -q /content

if not (ROOT / "pretrain.py").exists():
    !git clone -b {BRANCH} {REPO_URL} trm-latent-history-attention

%cd -q {ROOT}
!git fetch origin
!git checkout {BRANCH}
!git pull --ff-only origin {BRANCH}

# Do NOT pip install requirements.txt (adam-atan2/triton fail on Colab Python 3.13).
!python -m pip install -q -r requirements-colab.txt
print("cwd", pathlib.Path.cwd())
assert pathlib.Path("pretrain.py").exists(), "Setup did not land in the repo root"

## Build deterministic data

Stay in the repo root (`/content/trm-latent-history-attention`). The next cell copies `data/` from Drive if you already saved it, otherwise it builds 900 train bases (64 augs), 100 dev, and 1000 test. If the builder fails, the traceback is from that script — not from missing JSON.

In [ ]:
# Run from the repo root. Do not `%cd trm-latent-history-attention` here:
# if you are already inside the clone, that looks for a nested folder and the
# builder never writes artifacts/ under the path this cell reads.
from pathlib import Path
import json, os, subprocess, sys

ROOT = Path("/content/trm-latent-history-attention")
if not (ROOT / "pretrain.py").exists():
    ROOT = Path.cwd()
assert (ROOT / "pretrain.py").exists(), f"Not in the study repo: {ROOT}"
os.chdir(ROOT)

DRIVE = Path("/content/drive/MyDrive/trm-study")
if DRIVE.exists():
    if (DRIVE / "data/sudoku-study-v1").exists() and not (ROOT / "data/sudoku-study-v1").exists():
        !cp -a /content/drive/MyDrive/trm-study/data ./
    if (DRIVE / "artifacts").exists() and not (ROOT / "artifacts/data").exists():
        !cp -a /content/drive/MyDrive/trm-study/artifacts ./
    if (DRIVE / "outputs").exists() and not (ROOT / "outputs/study").exists():
        !cp -a /content/drive/MyDrive/trm-study/outputs ./

manifest = ROOT / "artifacts/data/sudoku_study_v1_manifest.json"
if not manifest.exists():
    subprocess.check_call(
        [sys.executable, str(ROOT / "dataset/build_sudoku_baseline_v2.py")],
        cwd=ROOT,
    )
if not manifest.exists():
    raise FileNotFoundError(
        f"Dataset builder did not write {manifest}. The failure is in the "
        "builder output above, not this json.load line."
    )

data = json.loads(manifest.read_text())
print("cwd", ROOT)
print(data["counts"])
print(data["leakage_assertions"])

## Run one job or the suite

Start with dry-run. Change `VARIANT` and `SEED` for one of the 15 jobs. The full suite is serial and may take roughly 15 hours at the one-hour target.

In [ ]:
# One of 15 jobs: variants B0/B1/B2/B3/P1 × seeds 0/1/2.
# --dry-run only prints the command. Remove it to train.
from pathlib import Path
import os
os.chdir("/content/trm-latent-history-attention" if Path("/content/trm-latent-history-attention/pretrain.py").exists() else Path.cwd())

VARIANT, SEED = "P1", 0
!python experiments/run_study.py single --variant {VARIANT} --seed {SEED} --dry-run
# Train (T4: skip torch.compile; it warns on bfloat16):
# !python experiments/run_study.py single --variant {VARIANT} --seed {SEED} --override compile_model=false
# All 15 jobs, serial (~15h):
# !python experiments/run_study.py suite --override compile_model=false

## Resume a capped/interrupted run

In [ ]:
from pathlib import Path
import os
os.chdir("/content/trm-latent-history-attention" if Path("/content/trm-latent-history-attention/pretrain.py").exists() else Path.cwd())

# Continues the same VARIANT/SEED from runtime_cap.pt or the latest step_*.pt.
!python experiments/run_study.py resume --variant {VARIANT} --seed {SEED} --dry-run
# !python experiments/run_study.py resume --variant {VARIANT} --seed {SEED} --override compile_model=false

## Inspect outputs

`metrics.jsonl` includes train throughput/VRAM/runtime, dev metrics, best-checkpoint decisions, and plateau evidence.

In [ ]:
from pathlib import Path
import json, os
os.chdir("/content/trm-latent-history-attention" if Path("/content/trm-latent-history-attention/pretrain.py").exists() else Path.cwd())

run_dir = Path(f"outputs/study/colab/{VARIANT}-seed{SEED}")
print("run_dir", run_dir.resolve())
print("checkpoints", sorted(p.name for p in run_dir.glob("*.pt")))
metrics_file = run_dir / "metrics.jsonl"
if metrics_file.exists():
    records = [json.loads(line) for line in metrics_file.read_text().splitlines()]
    print(json.dumps(records[-1], indent=2))
else:
    print("No metrics yet; this is the training log, not the test score.")

# Test-set scoring (needs best_dev.pt). Seed 0 may be missing if that run crashed.
# for seed in (1, 2):
#     ckpt = f"outputs/study/colab/P1-seed{seed}/best_dev.pt"
#     !python experiments/evaluate_study.py --config config/experiment/sudoku_study_colab.yaml --checkpoint P1={ckpt} --data data/sudoku-study-v1 --split test --seed {seed} --interventions
# !python experiments/analyze_results.py --input results/study --output results/study/analysis